In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Importing essential libraries

import os
# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
import pylab as pl
from glob import glob

import numpy as np
import tensorflow as tf
# tf.get_logger().setLevel('INFO')
import pickle as pkl

tf.autograph.set_verbosity(0)

from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs
from sklearn.metrics import accuracy_score
from sklearn.metrics import top_k_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

from tensorflow.keras.layers import Input, Dense, Layer
from tensorflow.keras.losses import BinaryCrossentropy, CategoricalCrossentropy, MeanSquaredError
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model

from typing import List, Tuple
from tqdm import tqdm

np.random.seed(123)
tf.random.set_seed(1234)

import warnings
warnings.filterwarnings('ignore')
import more_itertools as mit

# from malware_detection_inference import MalwareDetection

In [ ]:
base_dir = '/kaggle/input/homogeneous-data/homogeneous/'
print(os.listdir(base_dir))

In [ ]:
source_dir = os.path.join(base_dir, 'source')
target_dir = os.path.join(base_dir, 'target')
conv_model_weights_dir = '/kaggle/input/model-weights/weights/resnet_50_224x224.h5'

In [ ]:
print(f"no. of images in source : {len(glob(source_dir + '/**/**/*'))}")
print(f"no. of images in target : {len(glob(target_dir + '/**/**/*'))}")

In [ ]:
class MalwareDetection:
    """
        This class is an inference for trained models on malware images data
    """
    def __init__(self, model_path : str, optimizer, loss_fn : str, 
                         metrics : List[str], input_shape : Tuple):
        self.model_path = model_path
        self.optimizer = optimizer
        self.loss = loss_fn
        self.metrics = metrics
        self.input_shape = input_shape
        self.model = load_model(self.model_path)
        self.classes = ["benign", "malicious"]

    def load_image(self, img_path):
        image = cv2.imread(img_path)
        image_resized = cv2.resize(image, self.input_shape)
        image = np.expand_dims(image_resized, axis = 0)
        print("image load successfully")
#         print(img_path)
        return image

    def check_malware_image(self, img_path):
        img = self.load_image(img_path)
        out = self.model.predict(img)[0]
        pred = self.classes[np.argmax(list(out))]
        return f"predicted class is : {pred}"

    def get_embeddings(self, img_path):
        """
            This method is used to get second last layers embeddings
        """
        img = self.load_image(img_path)
        extractor = Model(inputs = model.inputs,
                          outputs = [model.layers[-2].output])
        emb = extractor(img)
        return emb


In [ ]:
class MalwareImageGAN:
    def __init__(self, source_images_dir : str, target_images_dir : str,\
                    input_shape : Tuple, conv_model_path : str,\
                    n_steps : int = 2000, batch_size : int = 4):
        """This class is a GAN architecture for detecting and generating
            malware images

        Args:
            source_images_dir (str): directory for source images
            target_images_dir (str): directory for target images
            input_shape (tuple) : input shape of images
            conv_model_path (str): path for pretrained model on malware images
        """

        self.source_images_dir = source_images_dir
        self.target_images_dir = target_images_dir

        self.source_train_images = os.path.join(self.source_images_dir, 'train')
        self.source_test_images = os.path.join(self.source_images_dir, 'test')

        self.target_train_images = os.path.join(self.target_images_dir, 'train')
        self.target_test_images = os.path.join(self.target_images_dir, 'test')

        self.input_shape = input_shape
        self.conv_model_path = conv_model_path
        self.n_classes = 2

        self.latent_dim = 512
        self.optimizer = Adam(0.0002, 0.5)  # Adam(1e-5)
        self.batch_size = batch_size
        self.n_steps = n_steps
        self.class_mapper = {
            0 : [1, 0], 1 : [0, 1],
        }


    def conv_model(self):
        malwareConvNet = MalwareDetection(model_path = self.conv_model_path,
                          optimizer = Adam(learning_rate = 0.001),
                          loss_fn = 'sparse_categorical_crossentropy',
                          metrics = ['accuracy'],
                          input_shape = self.input_shape)
        return malwareConvNet.model

    def build_generator_S(self):
        print("\n== Build Generator S...")
        model = self.conv_model()
        G_S = Model(inputs = model.inputs, \
            outputs = [model.layers[-2].output], name = "Generator_S")
        return G_S

    def build_generator_T(self):
        print("\n== Build Generator T...")
        model = self.conv_model()
        G_T = Model(inputs = model.inputs, \
            outputs = [model.layers[-2].output], name = "Generator_T")
        return G_T

    def build_generator(self):
        print("\n== Build Generator...")

        inputs = Input(self.latent_dim)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_G1")(inputs)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_G2")(net)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_G3")(net)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_G4")(net)

        DIrep = Dense(units = self.latent_dim, activation = tf.nn.sigmoid, name = "DIrep")(net)
        G = Model(inputs = inputs, outputs = DIrep, name = "Generator")

        #Classifier
        inputs = Input(DIrep.shape)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_C1")(inputs)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_C2")(net)
        net = Dense(units = self.n_classes, activation = tf.nn.softmax, name = "C")(net)
        C = Model(inputs = inputs, outputs = net, name = "Classifier")

        return G, C

    def build_disciminator(self):
        print("\n== Build Discriminator...")

        inputs = Input(self.latent_dim)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_D1")(inputs)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_D2")(net)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_D3")(net)
        net = Dense(units = 400, activation = tf.nn.relu, name = "fc_D4")(net)

        net = Dense(units = self.n_classes, activation = tf.nn.softmax, name = "D")(net)
        D = Model(inputs = inputs, outputs = net, name = "Discriminator")
        return D

    def create_image_tensor(self):
        """
            This method is used to create image tensors for testing folder
        """
        S_test_images = []
        S_test_labels = []
        T_test_images = []
        T_test_labels = []
        for i, im in enumerate(glob(self.source_test_images + '/**/*')):
            if i > 100:
                break
            label = 0 if im.split('/')[-1].split('.')[0].startswith('benign') else 1     # images name must start with benign_(someinteger).png
                                                            # and malicious_(someinteger).png
            img = tf.io.read_file(im)
            tensor = tf.io.decode_image(img, channels = 3, dtype = tf.dtypes.float32)
            tensor = tf.image.resize(tensor, list(self.input_shape))
            S_test_images.append(tensor)
            S_test_labels.append(label)

        for i, im in enumerate(glob(self.target_test_images + '/**/*')):
            if i > 100:
                break
            label = 0 if im.split('/')[-1].split('.')[0].startswith('benign') else 1     # images name must start with benign_(someinteger).png
                                                            # and malicious_(someinteger).png
            img = tf.io.read_file(im)
            tensor = tf.io.decode_image(img, channels = 3, dtype = tf.dtypes.float32)
            tensor = tf.image.resize(tensor, list(self.input_shape))
            T_test_images.append(tensor)
            T_test_labels.append(label)

        S_test_images = tf.convert_to_tensor(S_test_images)
        S_test_labels = tf.convert_to_tensor(S_test_labels)
        T_test_images = tf.convert_to_tensor(T_test_images)
        T_test_labels = tf.convert_to_tensor(T_test_labels)
        return S_test_images, S_test_labels, T_test_images, T_test_labels


    def d_loss(self, yhat_source, yhat_target):
        y_source = np.tile([1,0], (yhat_source.shape[0], 1))
        y_target = np.tile([0,1], (yhat_target.shape[0], 1))

        bce = CategoricalCrossentropy(from_logits = False)
        return bce(y_source, yhat_source) + bce(y_target, yhat_target)

    def g_loss(self, yhat_source, yhat_target):
        #[0,1]
        #[0,1]
        # ...

        y_source = np.tile([0,1], (yhat_source.shape[0], 1))
        y_target = np.tile([1,0], (yhat_target.shape[0], 1))

        bce = CategoricalCrossentropy(from_logits = False)
        return bce(y_source, yhat_source) + bce(y_target, yhat_target)

    def c_loss(self, yhat_class_source, yhat_class_target, y_source, y_target):
        # source_weight = 0.5
        # target_weight = 1
        bce = CategoricalCrossentropy(from_logits = False)
        # return (source_weight*bce(y_source, yhat_class_source) + target_weight* bce(y_target, yhat_class_target))/(source_weight + target_weight)
        return bce(y_source, yhat_class_source) + bce(y_target, yhat_class_target) #weight-source. bce() + .../(ws+wtt)


    def train(self):
        D = self.build_disciminator()
        G_S = self.build_generator_S()
        G_T = self.build_generator_T()
        G, C = self.build_generator()

        S_batches = tf.keras.preprocessing.image_dataset_from_directory(self.source_train_images,
                                                      seed = 123,
                                                      image_size = self.input_shape,
                                                      batch_size = self.batch_size)

        T_batches = tf.keras.preprocessing.image_dataset_from_directory(self.target_train_images,
                                                      seed = 123,
                                                      image_size = self.input_shape,
                                                      batch_size = self.batch_size)

        S_test_images, S_test_labels, T_test_images, T_test_labels = self.create_image_tensor()

        S_batches = iter(S_batches)
        T_batches = iter(T_batches)
        S_batches = mit.seekable(S_batches)
        T_batches = mit.seekable(T_batches)

        optimizer = self.optimizer

        g_loss_weight = 1
        c_loss_weight = 1

        print('====Loss Weights====')
        print('g_loss_weight: {0}'.format(g_loss_weight))
        print('c_loss_weight: {0}'.format(c_loss_weight))

        def _train_step():

            # Get a batch of source and target unlabeled samples
            try:
                x_batch_source, y_batch_source = next(S_batches)
            except:
                S_batches.seek(0)
                x_batch_source, y_batch_source = next(S_batches)
            try:
                x_batch_target, y_batch_target = next(T_batches)
            except:
                T_batches.seek(0)
                x_batch_target, y_batch_target = next(T_batches)
                
            #Create feature selections
            feature_S = G_S(x_batch_source)
            feature_T = G_T(x_batch_target)

            #Create domain invariant mapping using the Generator
            DIrep_source_samples = G(feature_S)
            DIrep_target_samples = G(feature_T)

            # Calculate the Domain loss
            with tf.GradientTape(persistent = True) as tape_disc:
                #Predict the domain using the discriminator
                yhat_source = D(DIrep_source_samples)
                yhat_target = D(DIrep_target_samples)
                
                # Compute D loss
                d_loss_value = self.d_loss(yhat_source, yhat_target)

            # Given loss, compute and apply gradient for discriminator:
            d_gradients = tape_disc.gradient(d_loss_value, D.trainable_variables)
            optimizer.apply_gradients(zip(d_gradients, D.trainable_variables))


            try:
                x_batch_source, y_batch_source = next(S_batches)
            except:
                S_batches.seek(0)
                x_batch_source, y_batch_source = next(S_batches)
            try:
                x_batch_target, y_batch_target = next(T_batches)
            except:
                T_batches.seek(0)
                x_batch_target, y_batch_target = next(T_batches)
            

            with tf.GradientTape(persistent = True) as tape_gen:

                #Create feature selections
                feature_S = G_S(x_batch_source)
                feature_T = G_T(x_batch_target)

                #Create domain invariant mapping using the Generator
                DIrep_source_samples = G(feature_S)
                DIrep_target_samples = G(feature_T)


                #Predict the domain using the discriminator
                yhat_source = D(DIrep_source_samples)
                yhat_target= D(DIrep_target_samples)

                #Predict the class of the samples

                class_pred_source = C(DIrep_source_samples)
                class_pred_target = C(DIrep_target_samples)

                # Compute G loss
                g_loss_value = self.g_loss(yhat_source, yhat_target)
                # Compute C loss

                y_batch_source = np.array(y_batch_source)
                y_batch_target = np.array(y_batch_target)

#                 y_batch_source_dummy = [self.class_mapper[y_batch_source[0]], self.class_mapper[y_batch_source[1]]]
#                 y_batch_target_dummy = [self.class_mapper[y_batch_target[0]], self.class_mapper[y_batch_target[1]]]
                
                try:
                    y_batch_source_dummy = [self.class_mapper[y_batch_source[i]] for i in range(len(y_batch_source))]
                    y_batch_target_dummy = [self.class_mapper[y_batch_target[i]] for i in range(len(y_batch_target))]
                except:
                    print(len(y_batch_source))
                    print(len(y_batch_target))

                y_batch_source = tf.Variable(y_batch_source_dummy, dtype = tf.float32)
                y_batch_target = tf.Variable(y_batch_target_dummy, dtype = tf.float32)
 
                c_loss_value = self.c_loss(class_pred_source, class_pred_target,
                                      y_batch_source, y_batch_target)


                combined_loss_value = (g_loss_weight * g_loss_value + c_loss_weight * c_loss_value) / (g_loss_weight + c_loss_weight)

            c_gradients = tape_gen.gradient(c_loss_value, C.trainable_variables)
            gs_gradients = tape_gen.gradient(combined_loss_value, G_S.trainable_variables)
            gt_gradients = tape_gen.gradient(combined_loss_value, G_T.trainable_variables)
            g_gradients = tape_gen.gradient(combined_loss_value, G.trainable_variables)

            optimizer.apply_gradients(zip(gs_gradients, G_S.trainable_variables))
            optimizer.apply_gradients(zip(gt_gradients, G_T.trainable_variables))
            optimizer.apply_gradients(zip(g_gradients, G.trainable_variables))
            optimizer.apply_gradients(zip(c_gradients, C.trainable_variables))

            return G_S, G_T, G, C, D, g_loss_value, c_loss_value, d_loss_value, combined_loss_value

        for step in tqdm(range(self.n_steps)):
            generator_S, generator_T, generator, classifier, discriminator, g_loss_value, c_loss_value, d_loss_value, combined_loss_value = _train_step()

            if (step % 50) == 0:
                y_source_pred_test = classifier.predict(generator(generator_S(S_test_images))).argmax(1)
                y_target_pred_test = classifier.predict(generator(generator_T(T_test_images))).argmax(1)


                accuracy_source = accuracy_score(S_test_labels, y_source_pred_test)
                accuracy_target = accuracy_score(T_test_labels, y_target_pred_test)

                y_source_DI_test = generator(generator_S(S_test_images))
                y_target_DI_test = generator(generator_T(T_test_images))


                y_source_domain_pred = discriminator(y_source_DI_test).numpy().argmax(1)
                y_target_domain_pred = discriminator(y_target_DI_test).numpy().argmax(1)
                y_domain_pred = tf.concat([y_source_domain_pred, y_target_domain_pred], axis=0)
                #why it is 1,0?
                y_domain_source_real = np.array([1] * y_source_domain_pred.shape[0])
                y_domain_target_real = np.array([0] * y_target_domain_pred.shape[0])
                y_domain_real =  tf.concat([y_domain_source_real, y_domain_target_real], axis=0)
                # print((y_domain_pred.numpy() == 1).sum())
                # print((y_domain_real.numpy() == 1).sum())

                domain_pred_accuracy_source = accuracy_score(y_domain_source_real, y_source_domain_pred)
                domain_pred_accuracy_target = accuracy_score(y_domain_target_real, y_target_domain_pred)

                f1_source = f1_score(S_test_labels, y_source_pred_test, average = 'weighted')
                f1_target = f1_score(T_test_labels, y_target_pred_test, average = 'weighted')

                track_loss = '\nStep %4d ==>Comb_loss: %4.4f G_Loss: %4.4f C_Loss: %4.4f D_Loss: %4.4f \n Acc Source: %4.4f Acc Target: %4.4f F1 Source: %4.4f F1 Target: %4.4f \n Acc Domain Source: %4.4f  Acc Domain Target: %4.4f' % (
                                                            step, combined_loss_value, g_loss_value.numpy(), c_loss_value.numpy(), d_loss_value.numpy(), 
                                                            accuracy_source, accuracy_target, f1_source, f1_target,
                                                            domain_pred_accuracy_source, domain_pred_accuracy_target)
                print(track_loss)

        print('Training ended')

In [ ]:
malGAN = MalwareImageGAN(source_images_dir = source_dir, \
                            target_images_dir = target_dir, \
                            input_shape = (224, 224), \
                            conv_model_path = conv_model_weights_dir,
                            batch_size = 16)


try:
    malGAN.train()
except Exception as e:
    print(str(e))
    import traceback
    traceback.print_tb(e.__traceback__)